# DepthWizard v4 — data prep (Modal Notebook, **CPU**)

Download → pack → prime percentile bounds → push to the `depthwizard-data`
Volume. Run this **once**; `02_train.ipynb` reads what it leaves behind.

Nothing here is library code: every cell shells out to `V4_Kaggle/prepare_data.py`
and `V4_Kaggle/pack_gamus_png.py`. Flag rationale lives in
[`V4_Kaggle/README.md`](../V4_Kaggle/README.md) and is not repeated.

## Before you run a single cell

Set these in the notebook sidebar, or the cells below will stop you:

| Sidebar tab | Set to |
|---|---|
| Compute profile | **8 CPU, 32 GiB, no GPU** — a GPU here is pure waste |
| Compute profile | **Idle timeout ≥ 60 min** (default is 10) |
| Filesystem | Volume `depthwizard-data` → mounts at `/mnt/depthwizard-data` |
| Secrets | `dw-hf` (`HF_TOKEN`) — SynRS3D and DFC23 come from the Hub |
| Secrets | `dw-kaggle` (`KAGGLE_USERNAME`, `KAGGLE_KEY`) — GAMUS and DFC23 mirrors |

Expect roughly 1–2 h wall clock and a couple of dollars of CPU time.

**Packing never writes straight to the Volume.** `ShardWriter` writes through
`open_memmap` and trims by rewriting (`packed.py:52,88`) — the exact
random-write pattern FUSE is worst at. So each dataset packs to container-local
`/scratch`, gets its bounds primed there, and is copied to the Volume
sequentially. Raw downloads are deleted as soon as their pack succeeds, because
the container disk holds the ~36 GiB of packs *and* the raw archives at once.


In [ ]:
# 1. Code + deps.  requirements.txt deliberately omits torch, and no part of the
#    prepare path imports it — this stays a slim CPU install.
!git clone --depth 1 https://github.com/AbhayKale332/DepthWizard.git /root/DepthWizard 2>/dev/null || echo "already cloned"
%cd /root/DepthWizard/FineTunning/V4_Kaggle
!apt-get -qq update && apt-get -qq install -y rsync unzip libglib2.0-0
%uv pip install -r requirements.txt "kaggle>=1.6.0"
# If the %uv magic rejects a flag, the shell form works too:
#   !uv pip install --system -r requirements.txt "kaggle>=1.6.0"

In [ ]:
import json, os, shlex, shutil, subprocess, sys, time
from pathlib import Path

CODE    = "/root/DepthWizard/FineTunning/V4_Kaggle"
DATA    = "/mnt/depthwizard-data"      # Volume, attached in the sidebar
SCRATCH = "/scratch/dwdata"            # container-local; dies with the kernel


def sh(cmd, cwd=CODE):
    """Run and fail loudly. Never build cmd from anything but the literals here."""
    print(f"\n[dw] $ {shlex.join(cmd)}", flush=True)
    t = time.time()
    if subprocess.run(cmd, cwd=cwd, check=False).returncode:
        raise RuntimeError(f"[dw] failed after {(time.time() - t) / 60:.1f} min: {cmd[0]}")
    print(f"[dw] ok ({(time.time() - t) / 60:.1f} min)", flush=True)


def stores(root):
    """Print every store's tile count. Read these.

    v3 wrote a gamus/val holding 129 of 400 tiles and still reported success,
    because a tile whose download failed was swallowed as `skip: ...`.
    """
    n = 0
    print(f"\n[dw] stores under {root}:")
    for idx in sorted(Path(root).glob("*/*/index.json")):
        n += 1
        d = json.loads(idx.read_text())
        # .parents[1], not .parent — .parent is the split. That is the ba78dd8 bug.
        name = f"{idx.parents[1].name}/{idx.parent.name}"
        gib = sum(f.stat().st_size for f in idx.parent.glob("*.npy")) / 2**30
        ok = "bounds" if (idx.parent / "stretch_bounds_2_98.npy").exists() else "NO-BOUNDS"
        print(f"    {name:<20} n={d['n']:>6}  tile={d['tile_px']:>4}  "
              f"gsd={d['gsd_m']:<5}  seg={'yes' if d.get('has_seg') else 'no':<3}  "
              f"{gib:6.2f} GiB  {ok}")
    return n


DL = "/scratch/_dl"

assert Path(DATA).is_dir(), (
    f"{DATA} is not mounted — sidebar → Filesystem → attach Volume 'depthwizard-data'")
for k in ("HF_TOKEN", "KAGGLE_USERNAME", "KAGGLE_KEY"):
    assert os.environ.get(k), (
        f"{k} is not set — sidebar → Secrets → attach 'dw-hf' and 'dw-kaggle'")
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

for p in (SCRATCH, DL):
    Path(p).mkdir(parents=True, exist_ok=True)

free = shutil.disk_usage("/scratch").free / 2**30
print(f"[dw] /scratch free: {free:.0f} GiB  (packs ~36 GiB, plus the raw archive "
      f"being processed — the biggest is DFC23 at ~12 GiB)")
if free < 60:
    print("[dw] WARNING: tight. Raise the disk in the compute profile, or run the "
          "three dataset cells in separate kernel sessions.")

if any(Path(DATA).glob("*/*/index.json")):
    print("[dw] the Volume already holds stores; pulling them back so packers skip them")
    sh(["rsync", "-a", f"{DATA}/", f"{SCRATCH}/"], cwd="/")
    stores(SCRATCH)

In [ ]:
# 3. Helpers used by each dataset cell.
#
# prime(): percentile bounds MUST be written while the store is still on
# /scratch. DATA is read-only during training, and packed.py:240-245 fails soft
# and recomputes them in-process instead — a silent per-epoch tax.
_PRIME = f"""
import sys; sys.path.insert(0, "{CODE}")
from pathlib import Path
from dwdata.packed import PackedStore
for i in sorted(Path("{SCRATCH}").glob("*/*/index.json")):
    if (i.parent / "stretch_bounds_2_98.npy").exists(): continue
    print("priming:", i.parent, flush=True)
    PackedStore(i.parent).prime_stretch_bounds(2.0, 98.0, workers=8)
"""


def prime():
    sh([sys.executable, "-c", _PRIME])


def push():
    """Copy finished .npy shards to the Volume, sequentially. Idempotent.
    Notebook mounts persist on their own; the last cell verifies it."""
    sh(["rsync", "-a", "--info=progress2", f"{SCRATCH}/", f"{DATA}/"], cwd="/")


def kaggle_get(slug, dest, expect):
    if all((Path(dest) / e).exists() for e in expect):
        return print(f"[dw] {slug} already at {dest}")
    sh(["kaggle", "datasets", "download", "-d", slug, "-p", dest, "--unzip"], cwd="/")
    if missing := [e for e in expect if not (Path(dest) / e).exists()]:
        raise RuntimeError(f"[dw] {slug} unpacked without {missing}; layout changed. "
                           f"Top level: {sorted(p.name for p in Path(dest).iterdir())[:20]}")


print("helpers ready")

In [ ]:
# 4. GAMUS — from the Kaggle PNG mirror, not the gated HF repo.
#
# prepare_gamus issues one hf_hub_download per file (~13 200 requests) and earns
# a CAS 429 that it swallows per tile. Cost of the mirror: no class rasters, so
# gamus ends up has_seg=no. Re-upload it with classes/ and repack with --force.
src = f"{DL}/gamus"
kaggle_get("akashch1512/gamusdataset", src, ["train", "val"])
sh([sys.executable, "pack_gamus_png.py", "--src", src,
    "--out", SCRATCH,                 # never omit: the default is /kaggle/temp/dwdata
    "--depth_scale", "auto", "--train_tiles", "0", "--val_tiles", "0",
    "--prime_workers", "8"])
prime(); push()
sh(["rm", "-rf", src], cwd="/")       # free the raw mirror before the next dataset
stores(SCRATCH)

In [ ]:
# 5. SynRS3D — 4 archives: all three g1 stores (the only source reaching past
#    GAMUS's 0.66 m ceiling) plus one g05. _syn_family splits them per GSD.
sh([sys.executable, "prepare_data.py", "--data_root", SCRATCH,
    "--datasets", "synrs3d", "--synrs3d_archives", "4"])
prime(); push()
stores(SCRATCH)

In [ ]:
# 6. DFC23 Track 2.
src = f"{DL}/dfc23"
kaggle_get("abhaydkale232/dfc23-track2-height-estimation", src,
           ["track2/train/rgb", "track2/train/dsm"])
# Top-level rgb/dsm/sar are a byte-identical copy of track2/train (~5.5 of the
# 12 GB); SAR is unused, the store is 3-channel.
for junk in ("rgb", "dsm", "sar", "track2/train/sar", "track2/val", "track2_test_data"):
    sh(["rm", "-rf", f"{src}/{junk}"], cwd="/")
sh([sys.executable, "prepare_data.py", "--data_root", SCRATCH,
    "--datasets", "dfc23", "--dfc23_dir", f"{src}/track2/train",
    "--dfc23_tile", "512", "--dfc23_val_frac", "0.15",
    "--dfc23_max_height_m", "150"])    # DFC23_Track2_Data_Audit.md's value
prime(); push()
sh(["rm", "-rf", src], cwd="/")
stores(SCRATCH)

In [ ]:
# 7. Verify what actually landed on the Volume. This is the cell that matters.
n = stores(DATA)
print(f"\n[dw] {n} stores on the volume")
print("[dw] expect: gamus/train ~3453  gamus/val ~859  synrs3d_g1  synrs3d_g05"
      "  dfc23_g050/train ~1506  dfc23_g050/val ~266")
print("[dw] every line must end in 'bounds', never 'NO-BOUNDS'")

## Read the counts before you spend anything on an H100

A short store is not an error — every packer swallows a failed tile as
`skip: ...` and reports success anyway. v3 shipped a `gamus/val` holding 129 of
400 tiles this way.

- `gamus/train ~3453`, `gamus/val ~859`
- `synrs3d_g1`, `synrs3d_g05`
- `dfc23_g050/train ~1506`, `dfc23_g050/val ~266`
- every line ending `bounds`, not `NO-BOUNDS`

Anything materially short: delete that store off the Volume and re-run its cell.
Adding `--force` to the packer repacks in place.

Then stop this kernel — it bills while it idles — and open `02_train.ipynb`.
